# LLM judge check

Interactively test the LLM-as-judge stage (`langtrend/judge.py`) on individual papers from a processed week.

Run from within `notebooks/` (paths are relative, e.g. `../data/processed/...`).
Configuration comes from `../.env` — see `.env.example` at the repo root. The default backend is
Groq's free tier; point `LLM_JUDGE_BASE_URL` at a local Ollama server for quota-free testing.

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, "..")

from langtrend.judge import (
    assemble_context, build_messages, collect_target_languages,
    judge_paper, safe_paper_id, save_judge_record,
)
from langtrend.llm_client import LLMClientConfig, OpenAICompatClient

config = LLMClientConfig.from_env()
client = OpenAICompatClient(config)
client.ping()  # raises with an actionable message if the endpoint/key is bad
print(f"Judge: {config.model} @ {config.base_url} (rpm={config.rpm}, context<={config.max_context_chars} chars)")

Judge: gpt-oss-120b @ https://api.cerebras.ai/v1 (rpm=4, context<=12000 chars)


## Pick a week and load its detected papers

In [2]:
# WEEK = "20260518_to_20260525"  # <- change me
WEEK = "20260622_to_20260629"  # <- change me

week_dir = Path(f"../data/processed/weeks/{WEEK}")
detected_path = week_dir / f"arxiv_papers_{WEEK}_detected.jsonl"

records = []
with detected_path.open(encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            records.append(json.loads(line))
print(f"{len(records)} detected papers in {WEEK}")

300 detected papers in 20260622_to_20260629


## List papers with low-resource (class 0–4) detections

These are the interesting candidates — class-5-only papers (English/Chinese etc.) rarely need review.

In [ ]:
for record in records[:400]:  # bump the slice to see more
    targets = collect_target_languages(record, classes={0, 1, 2, 3, 4})
    if not targets:
        continue
    langs = ", ".join(f"{t['language']}({t['class']})" for t in targets)
    print(f"{safe_paper_id(record['paper_id']):<16} {record['paper']['title'][:70]:<72} {langs}")

## Inspect one paper: assembled context + prompt

In [ ]:
# PAPER_ID = "2605.22785v1"  # <- change me (bare safe_id from the list above)
# PAPER_ID = "2605.24079v1"  # <- change me (bare safe_id from the list above)
# PAPER_ID = "2606.26807v1"  # <- change me (bare safe_id from the list above)

# latest week
PAPER_ID = "2606.24937v1"  # <- change me (bare safe_id from the list above) mix
# PAPER_ID = "2606.23566v2"  # <- change me (bare safe_id from the list above) true
# PAPER_ID = "2606.29567v1"  # <- change me (bare safe_id from the list above) mostly FP
# PAPER_ID = "2606.28667v1"  # <- change me (bare safe_id from the list above) ASL, multi word

record = next(r for r in records if safe_paper_id(r["paper_id"]) == PAPER_ID)
targets = collect_target_languages(record)
context = assemble_context(record, week_dir, targets, max_chars=config.max_context_chars)
messages = build_messages(context, targets)

print(record["paper"]["title"])
print(f"targets: {[t['language'] for t in targets]}")
print(f"context: {context.total_chars} chars, coverage={context.coverage}, {len(context.snippets)} snippets")
print()

# Full prompt exactly as sent to the model — every message, in order.
for msg in messages:
    print(f"===== {msg['role'].upper()} MESSAGE =====")
    print(msg["content"])
    print()

## Judge the paper and compare against the regex detections

In [ ]:
judge_record = judge_paper(record, week_dir, client, config)

width = max(len(name) for name in judge_record["verdicts"]) if judge_record["verdicts"] else 10
print(f"model: {judge_record['judge_model']}  coverage: {judge_record['context_coverage']}")
print()
for target in targets:
    v = judge_record["verdicts"].get(target["language"])
    verdict = v["verdict"] if v else "(unjudged)"
    reason = v["reason"] if v else ""
    print(f"{target['language']:<{width}}  | class {target['class']}  | {verdict:<15} \n{reason}\n--------------------")